# 06 - Model Comparison

Side-by-side comparison of all models: GMM, SVM, CNN, ECAPA-TDNN.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.evaluation.visualization import plot_model_comparison

plt.style.use('seaborn-v0_8-paper')

In [ ]:
# Load all results
results = {}
model_dirs = {
    'GMM-UBM': 'baseline_gmm',
    'SVM': 'baseline_svm',
    'CNN': 'cnn_mel_spectrogram',
    'ECAPA-TDNN': 'ecapa_tdnn',
}

for display_name, dir_name in model_dirs.items():
    metrics_path = Path(f'../results/{dir_name}/metrics.json')
    if metrics_path.exists():
        with open(metrics_path) as f:
            results[display_name] = json.load(f)
        print(f'{display_name}: loaded')
    else:
        print(f'{display_name}: NOT FOUND')

print(f"\nLoaded {len(results)} models for comparison.")

In [ ]:
# Comparison table
rows = []
for name, metrics in results.items():
    rows.append({
        'Model': name,
        'Accuracy (%)': f"{metrics.get('accuracy', 0) * 100:.1f}",
        'Top-5 Accuracy (%)': f"{metrics.get('top5_accuracy', 0) * 100:.1f}",
        'EER (%)': f"{metrics.get('eer', 0) * 100:.2f}",
    })

df = pd.DataFrame(rows)
print("\n=== Model Comparison ===")
print(df.to_string(index=False))

In [ ]:
# Bar chart comparison
if len(results) >= 2:
    plot_model_comparison(results, save_path='../results/model_comparison.png')

In [ ]:
# Accuracy progression chart (showing improvement across methods)
if len(results) >= 2:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    models = list(results.keys())
    accuracies = [results[m].get('accuracy', 0) * 100 for m in models]
    colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db']
    
    bars = ax.bar(models, accuracies, color=colors[:len(models)], edgecolor='black', linewidth=0.5)
    
    for bar, acc in zip(bars, accuracies):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{acc:.1f}%', ha='center', fontsize=12, fontweight='bold')
    
    ax.set_ylabel('Accuracy (%)', fontsize=14)
    ax.set_title('Speaker Identification Accuracy by Model', fontsize=16)
    ax.set_ylim(0, 105)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/accuracy_progression.png', dpi=300)
    plt.show()